# trapX LLM Advisory — **`advisory_any_bar.py`** (any date + bar)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_any_bar_colab_v2.ipynb)

**Latest version.** This notebook runs the *actual* standalone `advisory_any_bar.py` tool — not a re-implementation. Given a **DATE** and **BAR_TIME** it:

1. Downloads that day's folder from the shared Google Drive (`gdown`, public, no Drive API) — the `llm_advisory_<date>.jsonl`, the trapX SQLite checkpoint, and the day CSVs.
2. **Gates** on the jsonl: stops if no pattern fired that minute.
3. Rebuilds the advisory input fresh (state-memory @ bar−1, the engine-computed snapshot per fired touchpoint, the minute's market data).
4. Fires **one converged LLM-advisory call** and prints the verdict.

The script + all skill prompts are **embedded below** (base64), so the notebook is fully self-contained — nothing else to clone.

**Backends:** `nvidia` (default, needs `NVIDIA_API_KEY`) or `anthropic` / `auto` (live-parity, needs `ANTHROPIC_API_KEY`).

## 1. Install deps + load API keys
Keys are read from Colab **Secrets** (🔑 left sidebar → add `NVIDIA_API_KEY` and/or `ANTHROPIC_API_KEY`, toggle notebook access). Set at least the one matching your chosen backend.

In [ ]:
!pip install -q gdown openai anthropic python-dotenv langgraph langgraph-checkpoint-sqlite

import os
try:
    from google.colab import userdata
    for _k in ('NVIDIA_API_KEY', 'ANTHROPIC_API_KEY'):
        try:
            _v = userdata.get(_k)
        except Exception:
            _v = None
        if _v:
            os.environ[_k] = _v.strip()
except Exception:
    pass

print('NVIDIA_API_KEY   :', 'set' if os.environ.get('NVIDIA_API_KEY') else 'MISSING')
print('ANTHROPIC_API_KEY:', 'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING')

## 2. Write the embedded `advisory_any_bar.py` + `skills/` to disk
Decodes the base64 payloads into `/content/advisory_any_bar.py` and `/content/skills/*.md`. The script's `rules_sha` drift check matches each fired touchpoint to these exact skill texts.

In [ ]:
import base64, json, pathlib

SCRIPT_B64 = "..."
SKILLS_B64 = "..."

pathlib.Path('advisory_any_bar.py').write_bytes(base64.b64decode(SCRIPT_B64))
skills = json.loads(base64.b64decode(SKILLS_B64).decode('utf-8'))
sd = pathlib.Path('skills'); sd.mkdir(exist_ok=True)
for name, text in skills.items():
    (sd / name).write_text(text, encoding='utf-8')
print('wrote advisory_any_bar.py +', len(skills), 'skill files to ./skills/')

## 3. Choose the bar + backend
`WHEN` is the bar in the script's native `DD-mon, HH:MM` form. `BACKEND`: `nvidia` (default), `anthropic`, or `auto` (pins to the backend/model the live engine recorded — like-for-like replay).

In [ ]:
#@title Parameters { run: "auto" }
WHEN    = "12-Jun, 09:19"  #@param {type:"string"}
BACKEND = "nvidia"         #@param ["nvidia", "anthropic", "auto"]
YEAR    = 2026             #@param {type:"integer"}
# Optional: model override (blank = backend default).
MODEL   = ""               #@param {type:"string"}
print('WHEN   =', WHEN)
print('BACKEND=', BACKEND, '  YEAR=', YEAR, '  MODEL=', MODEL or '(default)')

## 4. Mount Google Drive + resolve the day folder
Mounts your Drive and resolves the **day folder** for the chosen bar (`VM-4-logs/<Mon_DD>`, e.g. `Jun_12`). The notebook reuses the script's own date parser so the folder name always matches. This same folder is used as **both** the data source *and* the place the verbose log is saved — so the log lands in the exact Drive folder the date was read from.

In [ ]:
#@title Drive folder { run: "auto" }
VM_LOGS_DIR = "/content/drive/MyDrive/VM-4-logs"  #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob, argparse
sys.path.insert(0, '.')
import advisory_any_bar as aab           # the script we wrote in step 2

# Reuse the script's exact parser so the Drive folder name always matches.
req = aab.parse_request(argparse.Namespace(when=WHEN, date=None,
                                           time=None, year=YEAR))
FOLDER_NAME = req.mon_dd                  # e.g. 'Jun_12'
DAY_DIR  = f'{VM_LOGS_DIR}/{FOLDER_NAME}'
LOG_NAME = f"advisory_{req.yyyymmdd}_{req.time.replace(':', '')}.log"
LOG_OUT  = f'{DAY_DIR}/{LOG_NAME}'

os.makedirs(DAY_DIR, exist_ok=True)
HAS_LOCAL = bool(glob.glob(f'{DAY_DIR}/llm_advisory_*.jsonl'))
print(f'date {req.iso_date}  ->  Drive folder {FOLDER_NAME!r}')
print('DAY_DIR :', DAY_DIR)
print('source  :', 'Drive day folder (--local-dir)' if HAS_LOCAL
      else 'gdown download (folder has no jsonl yet)')
print('log_out :', LOG_OUT)

## 5. Run the advisory — log saved back to the Drive folder
Invokes `python advisory_any_bar.py "<WHEN>" --skills-dir skills --backend <BACKEND> --no-live --out <DAY_DIR>/advisory_<date>_<time>.log`. When the Drive folder already holds the day's `llm_advisory_*.jsonl` it reads straight from Drive (`--local-dir`); otherwise the script fetches the day via `gdown`. Either way **`--out` writes the verbose log into the same Drive folder** (auto-suffixed `_1/_2/…` so a re-run never overwrites).

> **`--no-live` is required on Colab.** The script normally switches to a *live* path when the requested date is **today**, reading market data from a local Postgres (`localhost:5433`) that only exists on the trapX VM — on Colab that connection is refused and the run exits *before* writing any log. `--no-live` forces the Drive/CSV path, so it works for today's date too (as long as the day folder holds the CSVs + checkpoint `.db` + jsonl).

In [ ]:
import subprocess, sys, shlex

# --no-live: Colab has no local Postgres, so always use the day folder's
# CSVs/DB/jsonl (the script would otherwise go 'live' when date==today).
cmd = [sys.executable, 'advisory_any_bar.py', WHEN,
       '--skills-dir', 'skills', '--backend', BACKEND, '--year', str(YEAR),
       '--no-live', '--out', LOG_OUT]
if HAS_LOCAL:
    cmd += ['--local-dir', DAY_DIR]
if MODEL.strip():
    cmd += ['--model', MODEL.strip()]
print('$', ' '.join(shlex.quote(c) for c in cmd), '\n')

proc = subprocess.run(cmd, capture_output=True, text=True)
# Progress / [STAGE] lines go to stderr; the verdict goes to stdout.
print(proc.stderr)
print('=' * 70)
print(proc.stdout)
if proc.returncode != 0:
    print('\n[exit code]', proc.returncode)

## 6. Confirm the log on Drive + show it
Lists the verbose log(s) now sitting in the Drive day folder and prints the newest one. This is the file saved back to `VM-4-logs/<Mon_DD>/` — persisted in Drive, not just the Colab VM.

In [ ]:
import glob, os
logs = sorted(glob.glob(f'{DAY_DIR}/advisory_{req.yyyymmdd}_*.log'),
              key=os.path.getmtime)
if logs:
    print(f'{len(logs)} verbose log(s) saved in Drive folder {DAY_DIR}:')
    for p in logs:
        print('   ', os.path.basename(p), f'({os.path.getsize(p)} bytes)')
    print('\n===== newest:', os.path.basename(logs[-1]), '=====\n')
    print(open(logs[-1], encoding='utf-8').read())
else:
    print('no verbose log in', DAY_DIR,
          '— did the run gate out (no pattern at that bar) or error?')